# Hybrid CNN-LSTM Stock Price Predictor Model

In [27]:
import pandas as pd
import numpy as np
from keras.layers import Input, Conv1D, Dense, BatchNormalization, Dropout, LSTM
from keras.models import Model

## Data

In [28]:
df = pd.read_csv("data/all_data.csv")
df.drop(columns=["date"], inplace=True)

# Create numerical representation of labels for model to use
labels = sorted(list(set(df["label"])))
labels_indices = dict((l, i) for i, l in enumerate(labels))
indices_labels = dict((i, l) for i, l in enumerate(labels))

# Down=0, Flat=1, Up=2  (alphabetical order)
df["label_num"] = df["label"].map(labels_indices)

In [29]:
X, y = [], []
features = df.drop(columns=["label", "label_num"]).values
label_nums = df["label_num"].values
window_size = 20
num_features = len(df.drop(columns=["label", "label_num"]).columns)

# Convert data into 3D for CNN (samples, timesteps, features)
for i in range(0, len(df) - window_size): 
        X.append(features[i : i + window_size]) # Add a 2D section of data of all features from the days within the current window size
        y.append(label_nums[i + window_size]) # Add the labels for each day in the window size

X = np.array(X)
y = np.array(y)

X.shape # Shape should be ((len(df) - window_size), window_size, num_features)
y.shape # Shape should be ((len(df) - window_size),)


(1187,)

In [30]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## Model

All numbers are not real yet

In [31]:
inputs = Input(shape=(window_size,num_features))

# CNN Block
conv1 = Conv1D(filters=64, kernel_size=3, activation='relu')(inputs)
conv1 = BatchNormalization()(conv1)
conv1 = Dropout(0.2)(conv1)

conv2 = Conv1D(filters=64, kernel_size=5, activation='relu')(conv1)
conv2 = BatchNormalization()(conv2)
conv2 = Dropout(0.2)(conv2)

# LSTM Block
lstm = LSTM(5)(conv2)

# Dense layers and output
dense1 = Dense(5, activation='relu')(lstm)
outputs = Dense(3, activation='softmax')(dense1)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 20, 66)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 18, 64)         │        12,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 18, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 14, 64)         │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 14, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 5)              │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 5)              │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,240 (137.66 KB)

 Trainable params: 34,984 (136.66 KB)

 Non-trainable params: 256 (1.00 KB)

In [32]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=128, epochs=15, validation_data=(X_test, y_test))

score = model.evaluate(X_train, y_train, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Epoch 1/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3256 - loss: 1.1079 - val_accuracy: 0.2773 - val_loss: 1.1042
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3604 - loss: 1.0952 - val_accuracy: 0.4034 - val_loss: 1.0912
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3741 - loss: 1.0913 - val_accuracy: 0.4076 - val_loss: 1.0898
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4025 - loss: 1.0834 - val_accuracy: 0.3782 - val_loss: 1.0947
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4078 - loss: 1.0819 - val_accuracy: 0.3866 - val_loss: 1.0993
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4226 - loss: 1.0781 - val_accuracy: 0.3613 - val_loss: 1.1043
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4173 - loss: 1.0731 - val_accuracy: 0.3571 - val_loss: 1.1006
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4257 - loss: 1.0672 - val_accuracy: 0.3739 - val_loss: 1.1006


In [33]:
import numpy as np

sequence = np.random.random((100,))
print(sequence)

a = [1, 2, 3, 5, 6, 6, 7]
a[::2]

[0.03393119 0.37289036 0.52147583 0.53640143 0.93674006 0.68784516
 0.44326092 0.43794825 0.03497373 0.39871551 0.39723925 0.22262261
 0.25038627 0.35974123 0.77379184 0.59161757 0.57904685 0.91071115
 0.13880777 0.28874696 0.70818327 0.29445045 0.52810567 0.93083785
 0.56562404 0.06846671 0.10815808 0.89505257 0.09167646 0.73137961
 0.06956837 0.96886371 0.76054651 0.96048105 0.19131356 0.47180523
 0.44475561 0.07423359 0.19212937 0.68879831 0.64718847 0.65258881
 0.23094321 0.00490767 0.99142537 0.90444023 0.05915879 0.25097872
 0.14970287 0.69641994 0.61850438 0.71336238 0.12374529 0.95779544
 0.58016035 0.06033432 0.77787498 0.18713804 0.29883309 0.60207034
 0.59827776 0.70586042 0.36291605 0.6112266  0.31268277 0.02426406
 0.85094921 0.65108498 0.80754895 0.60121686 0.68983837 0.37400053
 0.44725233 0.3397547  0.71144335 0.21143388 0.79435056 0.41533241
 0.23517513 0.80659424 0.77405022 0.09601591 0.45164725 0.4348079
 0.83417642 0.01372625 0.20123044 0.03989087 0.89887641 0.19936

[1, 3, 6, 7]

## Training Model